In [ ]:
import os
import re
import string
import zipfile
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# ==========================================
# 1. EXTRACT FROM SPECIFIED ZIP PATH
# ==========================================
zip_path = "/archive(1).zip"
extract_dir = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("Extraction completed successfully!")

# Set file paths based on the extracted structure
base_folder = os.path.join(extract_dir, "Genre Classification Dataset")
train_path = os.path.join(base_folder, "train_data.txt")
test_path = os.path.join(base_folder, "test_data_solution.txt")

# ==========================================
# 2. LOAD DATASET
# ==========================================
columns = ['ID', 'TITLE', 'GENRE', 'DESCRIPTION']

print("Loading dataset...")
train_df = pd.read_csv(train_path, sep=':::', engine='python', names=columns)
test_df = pd.read_csv(test_path, sep=':::', engine='python', names=columns)

# ==========================================
# 3. TEXT PREPROCESSING
# ==========================================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Cleaning text data...")
train_df['CLEAN_DESC'] = train_df['DESCRIPTION'].apply(clean_text)
test_df['CLEAN_DESC'] = test_df['DESCRIPTION'].apply(clean_text)

# ==========================================
# 4. TF-IDF VECTORIZATION
# ==========================================
print("Vectorizing text features...")
tfidf = TfidfVectorizer(stop_words='english', max_features=10000)

X_train = tfidf.fit_transform(train_df['CLEAN_DESC'])
y_train = train_df['GENRE'].str.strip()

X_test = tfidf.transform(test_df['CLEAN_DESC'])
y_test = test_df['GENRE'].str.strip()

# ==========================================
# 5. MODEL TRAINING & EVALUATION
# ==========================================
# Multinomial Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_preds = nb_model.predict(X_test)
print(f"\nMultinomial Naive Bayes Accuracy: {accuracy_score(y_test, nb_preds):.4f}")

# Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)
print(f"Logistic Regression Accuracy: {accuracy_score(y_test, lr_preds):.4f}")

print("\n--- Classification Report (Logistic Regression) ---")
print(classification_report(y_test, lr_preds, zero_division=0))

# ==========================================
# 6. CUSTOM PREDICTION
# ==========================================
def predict_genre(plot_summary):
    cleaned = clean_text(plot_summary)
    vec = tfidf.transform([cleaned])
    return lr_model.predict(vec)[0]

sample_plot = "A detective investigates a series of mysterious crimes across the city."
print(f"\nSample Plot: '{sample_plot}'")
print(f"Predicted Genre: {predict_genre(sample_plot)}")

Extraction completed successfully!
Loading dataset...
Cleaning text data...
Vectorizing text features...

Multinomial Naive Bayes Accuracy: 0.5199
Logistic Regression Accuracy: 0.5881

--- Classification Report (Logistic Regression) ---
              precision    recall  f1-score   support

      action       0.50      0.29      0.37      1314
       adult       0.61      0.23      0.34       590
   adventure       0.64      0.16      0.25       775
   animation       0.54      0.05      0.10       498
   biography       0.00      0.00      0.00       264
      comedy       0.54      0.59      0.56      7446
       crime       0.38      0.03      0.06       505
 documentary       0.67      0.86      0.75     13096
       drama       0.54      0.78      0.64     13612
      family       0.51      0.08      0.14       783
     fantasy       0.50      0.04      0.07       322
   game-show       0.91      0.49      0.64       193
     history       0.00      0.00      0.00       243
      